# 9.18 換一種說法後，還能用原本的信心決定接不接題？


「查卡片進度」換成「想追蹤製卡到哪一步」，人可能認為同義，模型的分數卻可能改。另一批客服紀錄也可能有不同省略方式。原資料上訂好的「九成以上就自動處理」，到新材料上還可靠嗎？必須保留規則，用新材料的正確答案核對。

前置是[9.9–9.10的信心與溫度校準](https://birdhackor.github.io/tiny-perceptron-vlm/9.9.html)、[9.13的資訊是否足夠](https://birdhackor.github.io/tiny-perceptron-vlm/9.13.html)、[9.14–9.16的資料分工與決策](https://birdhackor.github.io/tiny-perceptron-vlm/9.14.html)。以下以`T`記校準溫度，以`τ`記接受門檻。**校準資料**只選T，讓報出的機率盡量貼近成敗；**決策資料（policy split）**只選τ，決定哪些題可自動處理；**最終測試（final）**只評這套已固定的規則。不同用途不能一面看答案、一面反覆調到滿意。

這裡Choice問「使用者想辦哪個服務？」；缺編號時仍可能正確選card，但資訊不足，不能自動執行。因此接受規則讀的是模型輸出：校準後的最大候選機率`p_max >= τ`、選中的不是none，而且資訊充分的**預測**為yes。正確答案只在事後計分：已接受題若選錯、實際不支援，或實際資訊不足，任一成立都算錯誤。

**覆蓋率（coverage）**是接受數除以全部預定題數；**接受風險（risk）**是接受題中的錯誤數除以接受數。none與轉交仍在覆蓋率分母內。零題接受時，風險是未定義，不能印成零錯誤率，讓「全部不接」看起來特別可靠。

先修正一個容易擴張的溫度結論：正T保住的是**同一題內**的分數高低與argmax，並不保住不同題的`p_max`排序。不同題中其餘候選的距離也會影響softmax分母。下面先算出反例，再走完三份資料的分工。所有分數、正確答案與來源標籤都是人工假設紀錄，沒有執行語言模型。

程式以**負對數似然（negative log-likelihood，NLL）**記猜錯代價：對每題取正確答案機率的負自然對數，再平均。在這個分類問題中，它就是[1.8的分類交叉熵](https://birdhackor.github.io/tiny-perceptron-vlm/1.8.html)的平均；正解機率越低，代價越大。選T時降低這個代價，不是改掉最高分候選。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import math


def softmax(z, T=1.0):
    scaled = [x / T for x in z]
    exp = [math.exp(x - max(scaled)) for x in scaled]
    return [x / sum(exp) for x in exp]


for T in [1.0, 0.1]:
    x = max(softmax([1.0, 0.0, 0.0], T))
    y = max(softmax([0.5, 0.0, -10.0], T))
    print("跨題反例", T, "X", round(x, 5), "Y", round(y, 5))

# 五個假設的校準家族。原始分數相同，gold分別是0、0、0、1、2。
calibration = [([math.log(3), 0.0, 0.0], y) for y in [0, 0, 0, 1, 2]]
temperatures = [0.25 * 2 ** (k / 8) for k in range(33)]


def nll(T):
    return sum(-math.log(softmax(z, T)[y]) for z, y in calibration) / len(calibration)


T = min(temperatures, key=lambda t: (nll(t), abs(t - 1), t))
print("校準選T", T, "平均NLL", round(nll(T), 4))

ids = ["card", "address", "none"]  # 每筆人工分數固定按此順序


def row(z, gold, info_pred=True, info_gold=True, supported_gold=True):
    return dict(z=z, gold=gold, info_pred=info_pred, info_gold=info_gold, supported_gold=supported_gold)


policy = [
    row([4.0, 0.0, -2.0], 0),
    row([0.0, 4.0, -2.0], 1),
    row([3.0, 0.0, -2.0], 0),
    row([2.0, 0.0, -2.0], 1),
    row([0.0, 0.0, 4.0], 2),
    row([4.0, 0.0, -2.0], 0, info_pred=False, info_gold=False),
]
final = {
    "原措辭": [row([4.0, 0.0, -2.0], 0), row([0.0, 3.0, -2.0], 1), row([0.0, 0.0, 4.0], 2)],
    "新措辭": [
        row([4.0, 0.0, -2.0], 1),
        row([2.0, 0.0, -2.0], 0),
        row([4.0, 0.0, -2.0], 0, info_pred=True, info_gold=False),
    ],
}


def evaluate(rows, tau):
    accepted = errors = 0
    for r in rows:
        p = softmax(r["z"], T)
        best = min(range(len(ids)), key=lambda i: (-p[i], ids[i]))
        take = p[best] >= tau and ids[best] != "none" and r["info_pred"]
        if take:
            accepted += 1
            errors += best != r["gold"] or not r["supported_gold"] or not r["info_gold"]
    risk = errors / accepted if accepted else None
    return accepted, errors, accepted / len(rows), risk


taus = [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 0.975, 0.99]
feasible = []
for tau in taus:
    n, e, coverage, risk = evaluate(policy, tau)
    if risk is not None and risk <= 0.05:
        feasible.append((coverage, tau))
if not feasible:
    raise RuntimeError("決策資料上沒有可行門檻；不能強行部署")
# 覆蓋率最大者優先；同覆蓋率選較高門檻。之後固定，不讀final來調。
_, tau = max(feasible)
print("固定tau", tau)
for name, rows in [("policy", policy), *final.items(), ("全部final", sum(final.values(), []))]:
    n, e, coverage, risk = evaluate(rows, tau)
    print(
        name,
        "接受",
        n,
        "/",
        len(rows),
        "錯誤",
        e,
        "coverage",
        round(coverage, 4),
        "risk",
        None if risk is None else round(risk, 4),
    )
print("零接受", evaluate(policy, 1.0))

報表中的`*final.items()`把原措辭、新措辭兩組展開成各一列；`sum(final.values(), [])`從空清單開始，把兩組題目接成一份「全部final」清單，並非把機率加總。

關鍵輸出是：

```text
跨題反例 1.0 X 0.57612 Y 0.62245
跨題反例 0.1 X 0.99991 Y 0.99331
校準選T 1.0 平均NLL 0.9503
固定tau 0.9
policy 接受 3 / 6 錯誤 0 coverage 0.5 risk 0.0
原措辭 接受 2 / 3 錯誤 0 coverage 0.6667 risk 0.0
新措辭 接受 2 / 3 錯誤 2 coverage 0.6667 risk 1.0
全部final 接受 4 / 6 錯誤 2 coverage 0.6667 risk 0.5
零接受 (0, 0, 0.0, None)
```

反例裡，T=1時Y信心較高，T=0.1時變成X較高，兩題各自最高候選都沒變。校準例的T=1是這批人工資料選出的結果，不代表溫度永遠不用改；此例只算Choice，各家族等權，正式多型別聚合還須事先固定各型權重。決策資料上τ=0.9能接一半、零錯；新措辭卻同時出現高信心選錯，以及資訊不足被誤判yes。這時保留失敗與原規則；不能看完final再提高τ，仍把結果叫同一次最終測試。

正式資料先按情境家族切分，原題的翻譯、候選增減與換措辭留在同一份；不能靠換句話製造獨立題數。要聲稱適用新來源，須真的留出不同來源。同生成器只換seed，仍是同來源的重新抽樣；只有另按模板或組合分組，並排除各份資料之間的重疊，才可稱相應的模板或組合留出。分來源、語言、候選數與題型列接受數和風險，並以家族為單位估計接受風險的不確定區間，用範圍呈現有限樣本造成的估計波動；這幾題的零錯不保證未來錯誤率。已套過T₀的輸出若要重新擬合溫度，先用「已縮放分數×T₀」還原原始分數，不能重複縮放；原始分數的取得方式須依介面規格確認。

這不只出現在人工分數裡。本輪在[第19章的新版候選判斷器](https://birdhackor.github.io/tiny-perceptron-vlm/19.23.html)也觀察到同樣的風險：T與τ先由各自的校準／策略家族選好，固定為約0.6484與0.6，再看60個最終家族。下表是那次實測，不是上面六題的延伸，也沒有拿這60題重挑門檻：

| 最終測試項目 | 實際結果 | 可以怎樣解讀 |
| --- | --- | --- |
| Choice選對候選 | 39／60，校準前後相同 | 正T沒有改變同題的argmax |
| Choice平均NLL | 原始2.6901，校準後3.9624 | 這批新家族上，固定T反而提高猜錯的平均代價 |
| 固定規則接受的題目 | 接受53／60，其中19題錯 | coverage為88.3%，接受風險為19／53≈35.8% |

校準時的選擇有其資料依據，卻沒有保證另一批家族的機率更準。39題選對也不能替53次自動接受背書；其中還要核對資訊是否真的充分。那一版C-v2因此保留為研究候選，當時沒有取代舊版預設。後續[19.25](https://birdhackor.github.io/tiny-perceptron-vlm/19.25.html)的C-v3有新介面、訓練與獨立最後測試，並非把這批final的T或τ重調後重報。完整型別、分母與失敗見[本輪結果](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/omni-products/v1/results-summary.json)。這裡先用已學過的指標讀出限制，模型怎麼計算留到後面的架構與產品章節。

練習：只把新措辭最後一筆的`info_pred`改為False。先算覆蓋率與風險，再執行。資訊不足那筆會被擋下，但另一次高信心選錯仍在：資訊判斷、候選正確率與校準各有自己的責任。

來源：[Guo等人的溫度縮放，§4](https://proceedings.mlr.press/v70/guo17a/guo17a.pdf)、[Kev固定版`raw_row`及`fit_temperature`](https://github.com/jaredpalmer/kev/blob/2074cc44256200bb37540103fa4572901faa10f3/kev/metrics.py#L233-L295)、[同版risk/coverage計算](https://github.com/jaredpalmer/kev/blob/2074cc44256200bb37540103fa4572901faa10f3/kev/metrics.py#L103-L143)。本節以獨立policy選固定門檻；依當批正誤事後挑最高coverage的曲線，只能作診斷，不能當成已固定的部署規則。
